# Python Iterators and Generators Cheatsheet

Understanding iteration protocol and lazy evaluation.

## 1. Iterators Basics

In [ ]:
# Iterable vs Iterator
# Iterable: object that can return an iterator (has __iter__)
# Iterator: object that produces values (has __iter__ and __next__)

# List is iterable
my_list = [1, 2, 3]

# Get iterator from iterable
my_iter = iter(my_list)
print(f"List: {my_list}")
print(f"Iterator: {my_iter}")

# Use next() to get values
print(f"\nnext(): {next(my_iter)}")
print(f"next(): {next(my_iter)}")
print(f"next(): {next(my_iter)}")
# print(next(my_iter))  # StopIteration!

In [ ]:
# For loop uses iterator protocol internally
my_list = [1, 2, 3]

# This:
for item in my_list:
    print(item, end=' ')
print()

# Is equivalent to:
iterator = iter(my_list)
while True:
    try:
        item = next(iterator)
        print(item, end=' ')
    except StopIteration:
        break

In [ ]:
# Creating custom iterator
class CountUp:
    """Iterator that counts from start to end."""
    
    def __init__(self, start, end):
        self.current = start
        self.end = end
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current > self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

# Use the iterator
counter = CountUp(1, 5)
for num in counter:
    print(num, end=' ')

In [ ]:
# Custom iterable (returns new iterator each time)
class Range:
    """Custom range-like class."""
    
    def __init__(self, start, end):
        self.start = start
        self.end = end
    
    def __iter__(self):
        return RangeIterator(self.start, self.end)

class RangeIterator:
    def __init__(self, start, end):
        self.current = start
        self.end = end
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current >= self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

# Can iterate multiple times
my_range = Range(1, 4)
print(list(my_range))  # [1, 2, 3]
print(list(my_range))  # [1, 2, 3] - works again!

## 2. Generators Basics

In [ ]:
# Generator function - uses yield
def count_up(start, end):
    """Generator that counts from start to end."""
    current = start
    while current <= end:
        yield current  # Pause and return value
        current += 1

# Create generator
gen = count_up(1, 5)
print(f"Generator: {gen}")
print(f"Type: {type(gen)}")

# Iterate
for num in gen:
    print(num, end=' ')

In [ ]:
# Generator execution flow
def simple_gen():
    print("Start")
    yield 1
    print("After first yield")
    yield 2
    print("After second yield")
    yield 3
    print("End")

gen = simple_gen()
print(f"Created generator")
print(f"First next(): {next(gen)}")
print(f"Second next(): {next(gen)}")
print(f"Third next(): {next(gen)}")
# print(next(gen))  # Would print "End" then raise StopIteration

In [ ]:
# Generator expression (like list comprehension but lazy)
squares_gen = (x ** 2 for x in range(5))
print(f"Generator: {squares_gen}")
print(f"Values: {list(squares_gen)}")

## 3. Yield Variations

In [ ]:
# yield from - delegate to another generator
def inner_gen():
    yield 1
    yield 2
    yield 3

def outer_gen():
    yield 'a'
    yield from inner_gen()  # Delegate to inner
    yield 'b'

print(list(outer_gen()))

In [ ]:
# Flatten nested lists using yield from
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)  # Recursive
        else:
            yield item

nested = [1, [2, 3, [4, 5]], 6, [7, [8, 9]]]
print(list(flatten(nested)))

In [ ]:
# Generator with return value
def gen_with_return():
    yield 1
    yield 2
    return "Done!"

gen = gen_with_return()
print(next(gen))
print(next(gen))
try:
    next(gen)
except StopIteration as e:
    print(f"Return value: {e.value}")

## 4. Generator Methods

In [ ]:
# send() - send value into generator
def accumulator():
    total = 0
    while True:
        value = yield total
        if value is not None:
            total += value

acc = accumulator()
print(next(acc))       # Start generator, get 0
print(acc.send(10))    # Send 10, get 10
print(acc.send(20))    # Send 20, get 30
print(acc.send(5))     # Send 5, get 35

In [ ]:
# throw() - throw exception into generator
def my_gen():
    try:
        yield 1
        yield 2
        yield 3
    except ValueError:
        yield "Error caught!"

gen = my_gen()
print(next(gen))
print(gen.throw(ValueError))  # Throw exception

In [ ]:
# close() - close generator
def my_gen():
    try:
        yield 1
        yield 2
        yield 3
    finally:
        print("Cleanup!")

gen = my_gen()
print(next(gen))
gen.close()  # Triggers GeneratorExit and finally block

## 5. Practical Generator Examples

In [ ]:
# Read large file line by line
def read_lines(filename):
    """Read file line by line (memory efficient)."""
    with open(filename, 'r') as f:
        for line in f:
            yield line.strip()

# Create sample file
with open('sample.txt', 'w') as f:
    f.write("Line 1\nLine 2\nLine 3")

for line in read_lines('sample.txt'):
    print(line)

import os
os.remove('sample.txt')

In [ ]:
# Infinite sequence
def fibonacci():
    """Infinite Fibonacci sequence."""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

# Get first 10 Fibonacci numbers
fib = fibonacci()
first_10 = [next(fib) for _ in range(10)]
print(f"Fibonacci: {first_10}")

In [ ]:
# Pipeline of generators
def numbers(n):
    for i in range(n):
        yield i

def squared(nums):
    for n in nums:
        yield n ** 2

def evens(nums):
    for n in nums:
        if n % 2 == 0:
            yield n

# Chain generators
pipeline = evens(squared(numbers(10)))
print(list(pipeline))

In [ ]:
# Batching/chunking
def chunks(iterable, size):
    """Split iterable into chunks of given size."""
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:  # Yield remaining items
        yield batch

data = range(10)
for chunk in chunks(data, 3):
    print(chunk)

In [ ]:
# Sliding window
from collections import deque

def sliding_window(iterable, n):
    """Generate sliding windows of size n."""
    it = iter(iterable)
    window = deque(maxlen=n)
    
    # Fill initial window
    for _ in range(n):
        window.append(next(it))
    yield tuple(window)
    
    # Slide window
    for item in it:
        window.append(item)
        yield tuple(window)

data = [1, 2, 3, 4, 5, 6]
for window in sliding_window(data, 3):
    print(window)

## 6. itertools Module

In [ ]:
import itertools

# count - infinite counter
counter = itertools.count(start=1, step=2)
print(f"Count: {[next(counter) for _ in range(5)]}")

# cycle - infinite cycle
cycler = itertools.cycle(['A', 'B', 'C'])
print(f"Cycle: {[next(cycler) for _ in range(7)]}")

# repeat - repeat value
repeater = itertools.repeat('X', 3)
print(f"Repeat: {list(repeater)}")

In [ ]:
# chain - combine iterables
a = [1, 2, 3]
b = [4, 5, 6]
print(f"Chain: {list(itertools.chain(a, b))}")

# islice - slice iterator
data = range(100)
print(f"Islice: {list(itertools.islice(data, 5, 10))}")

# takewhile/dropwhile
numbers = [1, 2, 3, 4, 5, 4, 3, 2, 1]
print(f"Takewhile < 4: {list(itertools.takewhile(lambda x: x < 4, numbers))}")
print(f"Dropwhile < 4: {list(itertools.dropwhile(lambda x: x < 4, numbers))}")

In [ ]:
# groupby - group consecutive elements
data = [('A', 1), ('A', 2), ('B', 3), ('B', 4), ('A', 5)]
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(f"{key}: {list(group)}")

## Summary

| Concept | Description |
|---------|-------------|
| Iterable | Has `__iter__()`, can create iterator |
| Iterator | Has `__iter__()` and `__next__()` |
| Generator | Function with `yield` |
| Generator Expression | `(expr for x in iter)` |
| `yield` | Pause and return value |
| `yield from` | Delegate to sub-generator |
| `send()` | Send value into generator |
| `throw()` | Throw exception into generator |
| `close()` | Close generator |